In [1]:
import pandas as pd

df = pd.read_csv("returns_cleaned.csv")
df.shape

(60000, 36)

In [3]:
drop_cols = ['order_id', 'customer_id', 'abuse_type', 'abuse_label',
             'order_date', 'return_date']
df_model = df.drop(columns=drop_cols)
df_model.shape

(60000, 30)

In [4]:
df_model.select_dtypes(include='object').columns

Index(['customer_segment', 'country', 'platform', 'device_type',
       'payment_method', 'product_category', 'return_reason',
       'shipping_carrier'],
      dtype='object')

In [5]:
categorical_cols = ['customer_segment', 'country', 'platform', 'device_type',
                     'payment_method', 'product_category', 'return_reason',
                     'shipping_carrier']

df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
df_encoded.shape

(60000, 70)

In [6]:
X = df_encoded.drop(columns=['is_risky'])
y = df_encoded['is_risky']

X.shape, y.shape

((60000, 69), (60000,))

In [7]:
X.to_csv("X_features.csv", index=False)
y.to_csv("y_target.csv", index=False)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((48000, 69), (12000, 69))

In [9]:
!pip install xgboost -q

from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)
print("Training done")

Training done


In [10]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # risk score between 0 and 1

print("Done")

Done


In [11]:
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, roc_auc_score

print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Risky']))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00      8412
       Risky       1.00      1.00      1.00      3588

    accuracy                           1.00     12000
   macro avg       1.00      1.00      1.00     12000
weighted avg       1.00      1.00      1.00     12000

ROC-AUC: 1.0


In [12]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[8412    0]
 [   0 3588]]


In [13]:
import pandas as pd

importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

importance.head(10)

,feature,importance
9,return_rate_pct,0.985839
17,customer_support_contacts,0.008230
6,days_to_return,0.001838
42,payment_method_PayPal,0.001011
2,avg_order_value_usd,0.000843
0,age,0.000558
3,refund_amount_requested_usd,0.000474
19,wishlist_to_cart_time_hrs,0.000473
7,total_orders_lifetime,0.000460
5,discount_used,0.000201


In [14]:
df[['return_rate_pct', 'total_returns_lifetime', 'total_orders_lifetime', 'is_risky']].corr()

,return_rate_pct,total_returns_lifetime,total_orders_lifetime,is_risky
return_rate_pct,1.000000,0.788044,0.186060,0.918310
total_returns_lifetime,0.788044,1.000000,0.553073,0.651764
total_orders_lifetime,0.186060,0.553073,1.000000,0.056605
is_risky,0.918310,0.651764,0.056605,1.000000


In [15]:
X2 = X.drop(columns=['return_rate_pct'])

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y
)

model2 = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                        eval_metric='logloss', random_state=42)
model2.fit(X_train2, y_train2)

y_pred2 = model2.predict(X_test2)
y_pred_proba2 = model2.predict_proba(X_test2)[:, 1]

print(classification_report(y_test2, y_pred2, target_names=['Legitimate', 'Risky']))

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00      8412
       Risky       1.00      1.00      1.00      3588

    accuracy                           1.00     12000
   macro avg       1.00      1.00      1.00     12000
weighted avg       1.00      1.00      1.00     12000



In [16]:
importance2 = pd.DataFrame({
    'feature': X2.columns,
    'importance': model2.feature_importances_
}).sort_values('importance', ascending=False)

importance2.head(10)

,feature,importance
16,customer_support_contacts,0.698701
17,previous_dispute_count,0.121211
8,total_returns_lifetime,0.053841
12,tracking_number_valid,0.031420
3,refund_amount_requested_usd,0.028504
2,avg_order_value_usd,0.026123
15,multiple_accounts_flag,0.013319
6,days_to_return,0.013009
18,wishlist_to_cart_time_hrs,0.004619
7,total_orders_lifetime,0.003960


In [17]:
keep_cols = [
    'age', 'account_age_days', 'customer_segment', 'country', 'platform',
    'device_type', 'payment_method', 'product_category', 'avg_order_value_usd',
    'refund_amount_requested_usd', 'is_high_value_item', 'discount_used',
    'days_to_return', 'return_reason', 'shipping_carrier',
    'total_orders_lifetime', 'total_returns_lifetime', 'wishlist_to_cart_time_hrs',
    'is_risky'
]

df_v2 = df[keep_cols].copy()
df_v2.shape

(60000, 19)

In [18]:
categorical_cols2 = ['customer_segment', 'country', 'platform', 'device_type',
                      'payment_method', 'product_category', 'return_reason',
                      'shipping_carrier']

df_v2_encoded = pd.get_dummies(df_v2, columns=categorical_cols2, drop_first=True)

X3 = df_v2_encoded.drop(columns=['is_risky'])
y3 = df_v2_encoded['is_risky']

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3, y3, test_size=0.2, random_state=42, stratify=y3
)
X3.shape

(60000, 58)

In [19]:
model3 = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                        eval_metric='logloss', random_state=42)
model3.fit(X_train3, y_train3)

y_pred3 = model3.predict(X_test3)
y_pred_proba3 = model3.predict_proba(X_test3)[:, 1]

print(classification_report(y_test3, y_pred3, target_names=['Legitimate', 'Risky']))

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00      8412
       Risky       1.00      1.00      1.00      3588

    accuracy                           1.00     12000
   macro avg       1.00      1.00      1.00     12000
weighted avg       1.00      1.00      1.00     12000



In [20]:
importance3 = pd.DataFrame({
    'feature': X3.columns,
    'importance': model3.feature_importances_
}).sort_values('importance', ascending=False)

importance3.head(10)

,feature,importance
9,wishlist_to_cart_time_hrs,0.512539
8,total_returns_lifetime,0.257075
3,refund_amount_requested_usd,0.127044
7,total_orders_lifetime,0.060824
6,days_to_return,0.031374
2,avg_order_value_usd,0.010844
5,discount_used,0.000179
11,customer_segment_New,0.000122
1,account_age_days,0.000000
0,age,0.000000


In [21]:
from sklearn.tree import DecisionTreeClassifier

simple_model = DecisionTreeClassifier(max_depth=3, random_state=42)
simple_model.fit(X_train3, y_train3)
y_pred_simple = simple_model.predict(X_test3)

print(classification_report(y_test3, y_pred_simple, target_names=['Legitimate', 'Risky']))

              precision    recall  f1-score   support

  Legitimate       0.97      1.00      0.98      8412
       Risky       1.00      0.92      0.96      3588

    accuracy                           0.98     12000
   macro avg       0.98      0.96      0.97     12000
weighted avg       0.98      0.98      0.98     12000



Noise

In [22]:
import numpy as np

np.random.seed(42)
X3_noisy = X3.copy()

# 1. Add small random noise to numeric columns (real sensors/self-reported data is never perfectly precise)
numeric_cols = ['age', 'account_age_days', 'avg_order_value_usd', 'refund_amount_requested_usd',
                 'days_to_return', 'total_orders_lifetime', 'total_returns_lifetime',
                 'wishlist_to_cart_time_hrs']

for col in numeric_cols:
    noise = np.random.normal(0, X3_noisy[col].std() * 0.1, size=len(X3_noisy))
    X3_noisy[col] = X3_noisy[col] + noise

# 2. Randomly flip 5% of the labels (real-world labels are never 100% correctly tagged —
# e.g. a human reviewer mislabels a case, or a genuinely borderline return gets tagged wrong)
y3_noisy = y3.copy()
flip_idx = np.random.choice(y3_noisy.index, size=int(0.05 * len(y3_noisy)), replace=False)
y3_noisy.loc[flip_idx] = 1 - y3_noisy.loc[flip_idx]

print("Noise added")

Noise added


In [23]:
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
    X3_noisy, y3_noisy, test_size=0.2, random_state=42, stratify=y3_noisy
)

model_noisy = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                             eval_metric='logloss', random_state=42)
model_noisy.fit(X_train_n, y_train_n)

y_pred_n = model_noisy.predict(X_test_n)
y_pred_proba_n = model_noisy.predict_proba(X_test_n)[:, 1]

print(classification_report(y_test_n, y_pred_n, target_names=['Legitimate', 'Risky']))

              precision    recall  f1-score   support

  Legitimate       0.94      0.97      0.96      8182
       Risky       0.94      0.87      0.91      3818

    accuracy                           0.94     12000
   macro avg       0.94      0.92      0.93     12000
weighted avg       0.94      0.94      0.94     12000



In [24]:
cm_noisy = confusion_matrix(y_test_n, y_pred_n)
print(cm_noisy)

[[7972  210]
 [ 479 3339]]


In [25]:
import joblib
import json

joblib.dump(model_noisy, "return_risk_model.pkl")
with open("model_columns.json", "w") as f:
    json.dump(list(X3_noisy.columns), f)

print("Saved final model and columns")

Saved final model and columns
